In [4]:
import numpy as np
import pandas as pd

Preparing a dataset with 10K values

In [5]:
num_samples = 10000

Data generation

In [6]:
data = {
    'search_frequency': np.random.uniform(0,1, num_samples),
    'insert_frequency': np.random.uniform(0,1, num_samples),
    'delete_frequency': np.random.uniform(0,1, num_samples),

    #Basic Constraints:
    'requires_ordering': np.random.choice([0,1], num_samples),
    'LIFO_behavior': np.random.choice([0,1], num_samples, p=[0.9,0.1]),
    'FIFO_behavior': np.random.choice([0,1], num_samples, p=[0.9,0.1]),
    'memory_constraint': np.random.choice([0,1, num_samples]),#Is the algorithm running in an environment with limited RAM (like embedded systems or edge devices) where memory efficiency is critical?

    #Tree/Graph Constraints:

    #natural parent-child relationship and not a flat list
    'is_hierarchical': np.random.choice([0,1], num_samples, p=[0.7,0.3]),

    #many-to-many relationships and not a strict top-down order, Eg:social networks
    'is_interconnected': np.random.choice([0,1], num_samples, p=[0.8,0.2]),

    #is there specific need to calculate the shortest route between two data points {eg: GPS navigation]
    'requires_pathfinding': np.random.choice([0,1], num_samples, p=[0.85,0.15]),

    #Is the data proceesed on priority instead of chronological order
    'requires_priority_processing': np.random.choice([0,1], num_samples, p=[0.9,0.1]),

    #Is the data massive and needs to be processed on external disks instead ram
    'is_massive_disk_based': np.random.choice([0,1], num_samples, p=[0.8,0.2]),

    #Complexity Constraints:

    #The system cannot tolerate anything slower than constant time O(1) for searching
    'requires_01_lookups': np.random.choice([0,1], num_samples, p=[0.85,0.15]),

    #The user needs to frequently find all elements between value two values in O(log n) time
    'requires_range_queries': np.random.choice([0,1], num_samples, p=[0.7,0.3]),

    #The user is willing to sacrifice execution speed to guarantee minimal RAM usage
    'prioritize_space_over_time': np.random.choice([0,1], num_samples, p=[0.6,0.4]),

    #Favors Linked Lists over arrays
    'requires_fast_dynamic_resizing': np.random.choice([0,1], num_samples, p=[0.7,0.3])
}

In [12]:
df = pd.DataFrame(data)
df

,search_frequency,insert_frequency,delete_frequency,requires_ordering,LIFO_behavior,FIFO_behavior,memory_constraint,is_hierarchical,is_interconnected,requires_pathfinding,requires_priority_processing,is_massive_disk_based,requires_01_lookups,requires_range_queries,prioritize_space_over_time,requires_fast_dynamic_resizing
0,0.172342,0.826694,0.314082,1,0,0,1,0,0,0,0,0,0,1,0,0
1,0.489333,0.834058,0.233071,1,0,0,1,0,0,0,0,0,0,1,0,0
2,0.287315,0.978841,0.154687,0,0,0,1,1,1,1,0,0,1,0,0,0
3,0.323217,0.426609,0.358982,0,0,0,1,1,1,1,0,0,0,1,0,0
4,0.991831,0.225462,0.652343,1,0,1,1,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.212766,0.443125,0.068983,0,0,0,1,0,0,0,1,0,0,1,1,0
9996,0.744975,0.216914,0.406219,1,0,0,1,0,0,0,0,0,0,0,0,1
9997,0.295658,0.658142,0.936350,0,0,0,1,0,0,0,0,0,0,0,1,1
9998,0.531089,0.466169,0.899162,1,0,0,1,0,0,0,0,0,0,0,0,0


Function for data labeling

In [14]:
def determine_structure(row):
    #Strictly O(1)-(Overrides almost everything)
    if row['requires_01_lookups']==1:
        if row['requires_range_queries']==1:
            return 'B-Tree' if row['prioritize_space_over_time']== 1 else 'AVL Tree'
        return 'hash table'

    #Range Queries: O(log n) structures
    if row['requires_range_queries']==1:
        if row['is_hierarchical']==1:
            return 'B+ Tree'
        return 'Binary Search Tree' if row['prioritize_space_over_time']==1 else 'AVL Tree'

    #Dynamic Resizing without O(n) latency spikes: Linked lists over arrays
    if row['requires_fast_dynamic_resizing'] == 1:
        if row['search_frequency'] < 0.3: 
         return 'Singly Linked List' if row['prioritize_space_over_time'] == 1 else 'Doubly Linked List'

    #Graphs & Networks (space/time trade-offs)
    if row['is_interconnected'] == 1:
        return 'Graph (Adjacency List)' if row['prioritize_space_over_time'] == 1 else 'Graph (Adjacency Matrix)'

    #Fallback Options
    if row['prioritize_space_over_time'] == 1:
        return 'Static Array'
        
    if row['search_frequency'] > 0.7:
        return 'Hash Table'
        
    return 'Dynamic Array'
    

In [15]:
df['target_data_structure'] = df.apply(determine_structure, axis=1)

In [16]:
df.to_csv('complexity_based_dsa_dataset.csv', index=False)